# Phase 5: TUV-x TS1/TSMLT Photolysis

Verifies upgraded TUV-x photolysis rates:
- Photolysis rates for Chapman reactions match Phase 2
- Alias mapping routes TUV-x labels to MICM parameters correctly
- Photolysis rate profiles are physically reasonable

**Pre-requisite:**
- `data/jw_480km_tuvx/output.nc` (Phase 5 run)
- `data/jw_480km_chapman/output.nc` (Phase 2 reference)

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data"
TUVX = DATA_DIR / "jw_480km_tuvx" / "output.nc"
PHASE2 = DATA_DIR / "jw_480km_chapman" / "output.nc"

## 1. Photolysis Rate Profiles

J-values as a function of altitude at a dayside cell.
Should increase with altitude due to less absorption.

In [ ]:
if TUVX.exists():
    ds = nc.Dataset(TUVX)
    # TODO: Update with actual photolysis rate diagnostic variable names
    j_vars = [v for v in ds.variables if v.startswith("j_") or v.startswith("J_")]
    if j_vars:
        # Pick a dayside, mid-latitude cell
        lat = np.degrees(ds["latCell"][:])
        lon = np.degrees(ds["lonCell"][:])
        dayside = (lon > -90) & (lon < 90)
        midlat = (lat > 30) & (lat < 60)
        candidates = np.where(dayside & midlat)[0]
        i_cell = candidates[0] if len(candidates) > 0 else 0

        fig, ax = plt.subplots(figsize=(8, 6))
        for jv in j_vars:
            profile = ds[jv][-1, i_cell, :]
            ax.plot(profile, range(len(profile)), label=jv)
        ax.set_xlabel("J-value (s⁻¹)")
        ax.set_ylabel("Level index")
        ax.set_title(f"Photolysis Rate Profiles at cell {i_cell}")
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No photolysis rate diagnostics found")
    ds.close()
else:
    print("Phase 5 output not found")

## 2. Regression Against Phase 2

Chapman photolysis rates should match between Phase 2 and Phase 5
(same reactions, upgraded TUV-x should give identical rates for Chapman subset).

In [ ]:
if TUVX.exists() and PHASE2.exists():
    ds5 = nc.Dataset(TUVX)
    ds2 = nc.Dataset(PHASE2)
    # Compare O3 field as proxy for photolysis rate correctness
    if "O3" in ds5.variables and "O3" in ds2.variables:
        diff = np.abs(ds5["O3"][:] - ds2["O3"][:]).max()
        print(f"Max |O3 diff| between Phase 5 and Phase 2: {diff:.2e}")
    ds5.close()
    ds2.close()
else:
    print("Need both Phase 2 and Phase 5 output for regression")